# Escritura de mensajes en Kafka

Este notebook tiene como objetivo insertar lotes de mensajes cada cierto tiempo en el tópico call-event de un clúster de Kafka gestionado en su totalidad por la empresa Confluent, a través 
de un productor idempotente o transaccional que evite la re-escritura de mensajes, esto es, se evita la inserción de mensajes ya presentes en el clúster ante un fallo en la inserción o continuación de este. El código total se divide en etapas o pasos para mejor comprensión del proceso:

In [12]:
# Importación de librerias necesarias
import pandas as pd
import json
import numpy as np
import time
from confluent_kafka import Producer

## 0. Generación de fichero total de mensajes de llamadas 

Elaboraremos un código que lee los tres ficheros de llamadas, concatena, ordena por fecha, y escribe en un directorio local


In [13]:
# definición de paths de ficheros de llamdas y path de fichero final con todos los mensajes a insertar
path_fichero_total_mensajes_kafka = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Mensajes a insertar en Kafka\mensajes_insertar_kafka.parquet"

path_calls_mh = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Respaldo\Michael H\Fct Call Full MH.csv"
path_calls_rb = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Respaldo\Rushern Baker\Fct Call Full RB.csv"
path_calls_vk = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Data Call Center Respaldo\Vin Kruttiventi\Fct Call Full VK.csv"

In [ ]:
# Concatenación simple de ficheros
lista_dfs_calls = [pd.read_csv(path_calls_vk), pd.read_csv(path_calls_rb), pd.read_csv(path_calls_mh)]
df_calls_total = pd.concat(lista_dfs_calls, ignore_index=True)
df_calls_total.head(5)

In [ ]:
# Revisamos esquema de dataframe antes de escribir para verificar tipología de campos
df_calls_total.info()

In [ ]:
# Escritura en disco de df_calls_total
df_calls_total.to_parquet(path_fichero_total_mensajes_kafka ,index=False)

## 1. Lectura de fichero de llamadas total de la muestra de datos para las tres campañas políticas 

El fichero generado en la etapa anterior, se carga en memoria y se ordenan los registros en orden ascendente según el campo **call_date**

In [14]:
# Definición de paths
path_fichero_total_llamadas = r"C:\Users\TPG\Documents\Trabajos UNIR\Semestre II\TFM\Mensajes a insertar en Kafka\mensajes_insertar_kafka.parquet"

In [15]:
# Lectura de fichero parquet
print("Leyendo datos del CSV ...")
df_total_llamadas = pd.read_parquet(path_fichero_total_llamadas)

Leyendo datos del CSV ...


In [16]:
# Verificación de carga correcta
display(df_total_llamadas.head(5), df_total_llamadas.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 436624 entries, 0 to 436623
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   call_date       436624 non-null  object 
 1   full_name       436624 non-null  object 
 2   list_id         436624 non-null  int64  
 3   gmt_offset_now  436624 non-null  float64
 4   comments        436624 non-null  object 
 5   length_in_sec   436453 non-null  float64
 6   alt_dial        436624 non-null  object 
 7   list_name       436624 non-null  object 
 8   status_name     436624 non-null  object 
 9   custom_fields   2371 non-null    object 
 10  Yard Sign       1159 non-null    object 
 11  Top_Issue       712 non-null     object 
 12  New Calls       293485 non-null  object 
dtypes: float64(2), int64(1), object(10)
memory usage: 43.3+ MB


,call_date,full_name,list_id,gmt_offset_now,comments,length_in_sec,alt_dial,list_name,status_name,custom_fields,Yard Sign,Top_Issue,New Calls
0,10/30/2024 20:47,Outbound Auto Dial,363,-7.0,1e84ef2fcc1772518e47502573cc189d2284377931bdc6...,0.0,NONE,candidate_name_3,Disconnected Number Auto,None,None,None,None
1,10/30/2024 20:47,Outbound Auto Dial,363,-7.0,1a6238fd0a23c2542e830e68d0339bdcb6796a7951f4b1...,0.0,NONE,candidate_name_3,Disconnected Number Auto,None,None,None,None
2,10/30/2024 20:47,Outbound Auto Dial,363,-7.0,399c132bf4e441e7785e1657a7820b2be455d38ce92eb3...,0.0,NONE,candidate_name_3,Disconnected Number Auto,None,None,None,None
3,10/30/2024 20:47,Outbound Auto Dial,363,-7.0,14fd2473303bd325cb5643901f5fcbbbcebac066212dd8...,0.0,NONE,candidate_name_3,Disconnected Number Auto,None,None,None,None
4,10/30/2024 20:47,Outbound Auto Dial,363,-7.0,5d7f7b414b3933a99304bb5d7015c4e67b03b74b7b6255...,0.0,NONE,candidate_name_3,Disconnected Number Auto,None,None,None,None


None

In [22]:
df_total_llamadas["status_name"].unique()

array(['Disconnected Number Auto', 'Answering Machine',
       'Answering Machine Msg Played', 'Lead Being Called',
       'Answering Machine SentToMesg', 'No Answer AutoDial', 'Hang Up',
       'Not Interested', 'Busy Auto', 'Support ONLY', 'Ringing Back',
       'Call Back', 'Already Voted', 'No Answer', 'Considering',
       'Dead Air', 'Outbound Pre-Routing Drop', 'Spanish Speaker',
       'Wrong Number', 'Busy', 'Undecided', 'Foreign Language',
       'Agent Not Available', 'Not Supporting', 'Remove From List',
       'DO NOT CALL', 'No Response', 'Moved', 'Disconnected Number',
       'Answering Machine Auto', 'Will Send', 'Hung Up', 'Support',
       'Support with Sign', 'Inbound Queue Timeout Drop', 'Brock Myers',
       'Lead To Be Called', 'Christopher Harrison', 'Declined Sale',
       'Aisha Braveboy', 'Rushern Baker'], dtype=object)

## 2. Creación de lista de cadenas JSON para insertar en Kafka 

El objetivo de este paso consiste en serializar el dataframe de modo que cada uno de sus registros, se convierta a una cadena json que puede escribirse en el tópico de Kafka **call-event**

In [ ]:
# Creamos la lista de registros y la serializamos correctamente
records  = df_total_llamadas.to_dict(orient="records") # Se transforma el dataframe a una lista de diccionarios de los registros
mensajes = [json.dumps(rec, ensure_ascii=False) for rec in records] # Cada diccionario se convierte a una cadena json

**Nota**: Es fundamental que en la cadena json los nulos aparezcan como *null*. Dado que los mensajes se han generado de la lectura de un fichero parquet, los valores nulos han sido correctamente codificados a null. Esto garantiza que al formatear la cadena json en Spark, el motor interprete correctamente los valores nulos

In [ ]:
# Mostramos los 3 primeros mensajes para verificar una serialización correcta
mensajes[:3]

## 3. Inserción de cadenas JSON al tópico **call-event**

A través de un productor idempotente o transaccional, se tiene como objetivo insertar mensajes en el tópico

In [ ]:
mensajes_prueba = mensajes[:100]

In [ ]:
# Funciones necesarias
def read_config():
  # Lee la configuración del cliente de client.properites del actual directorio de trabajo
  # y retorna un diccionario de configuraciones
  config = {}
  with open("client.properties") as fh:
    for line in fh:
      line = line.strip()
      if len(line) != 0 and line[0] != "#":
        parameter, value = line.strip().split('=', 1)
        config[parameter] = value.strip()
  return config


def produce(config):
  # creates a new producer instance
  producer = Producer(config)
  return producer

def insertar_kafka(producer, topic, messages, size_batch, ts):
    """
    Inserta en Kafka los mensajes de las filas del fichero parquet.
    Cada fila es un mensaje formateado como JSON. 
    """
    
    print("Insertando mensajes en el topic call-event...")
    last_idx = len(messages)-1
    for i, msj in enumerate(messages):
        producer.produce(topic, value=msj)
        if ((i + 1) % size_batch == 0) or (i == last_idx):
            print(f"Se ha insertado el mensaje {i} con valor {msj}")
            # send any outstanding or buffered messages to the Kafka broker
            producer.flush()
            time.sleep(ts)

def main():
  tam_lote = 3 # Tamaño del lote de mensajes (cambiar a 10000)
  tiempo_espera_lotes = 5 # Tiempo de espera entre inserción de lotes
  topico = "call-event" # Nombre del tópico a leer
  config = read_config() # Obtención de diccionario de configuración para conectarse al tópico
  productor = produce(config)

  # Inserción de mensajes personalizada
  insertar_kafka(productor, topico, mensajes_prueba, tam_lote, tiempo_espera_lotes)

**Estado**: El código se ha probado y funciona correctamente

In [ ]:
# Ejecución del proceso de inserción de mensajes en el clúster de Kafka de Confluent Cloud
main()

## Funciones adicional: Creación y borrado de tópicos del clúster de Kakfa

En caso de ser necesario, para agilizar procesos pueden usarse las funciones que siguen para gestionar la creación y borrado de tópicos del clúster de Confluent Kafka

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

def crear_topic(admin_client: AdminClient, topic_name: str, num_partitions: int = 6):
    """
    Crea el topic llamado topic_name con una sola partición y factor replicación 1.
    No devuelve nada pero mostrará un mensaje indicando si ha ido bien o no.
    
    :param admin_client: objeto AdminClient con el cliente de Kafka ya configurado
    :param topic_name: string con el nombre del topic que deseamos crear
    """
    topic = NewTopic(topic_name, num_partitions, replication_factor=1)
    fs = admin_client.create_topics([topic])
    for topic, f in fs.items():
        try:
            f.result()
            print(f"Topic {topic_name} creado")
        except Exception as e:
            print(f"Fallo al crear el topic {topic_name}: {e}")

def borrar_topic(admin_client: AdminClient, topic_name: str):
    """
    Borra un topic existente llamado topic_name que debe existir.
    No devuelve nada pero mostrará un mensaje indicando si ha ido bien o no.
    
    :param admin_client: objeto AdminClient con el cliente de Kafka ya configurado
    :param topic_name: string con el nombre del topic que deseamos borrar
    """
    fs = admin_client.delete_topics([topic_name])
    for topic, f in fs.items():
        try:
            f.result()
            print(f"Topic {topic_name} borrado")
        except Exception as e:
            print(f"Fallo al borrar el topic {topic_name}: {e}")

def main():
    topico = "example-1" # Definición de nombre de tópico
    numero_particiones = 6 # Valor por defecto en Confluent Kafka
    config = read_config() # Obtención de diccionario de configuración para conectarse al tópico
    admin_client = AdminClient(conf) # Creación de cliente administrado para habilitar la creación y borrado de tópicos
    crear_topic(admin_client, topico, numero_particiones) # Creación de tópico con el nombre especificado
    borrar_topic(admin_client, topico) # Supresión de tópico indicado